# 1D Convolution

In [1]:
!nvidia-smi

Fri Jan 23 13:25:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [14]:
%%writefile utils.h
/**
 * @file utils.h
 * @brief Utility functions for CUDA development
 *
 * This header provides core utility functions focusing on error checking timing,
 * initialising arrays in memory and result comparison.
 */

#ifndef UTILS_H
#define UTILS_H

#include <iostream>
#include <chrono>
#include <cstdlib>
#include <ctime>
#include <cmath>
#include <cuda_runtime.h>

/**
 * @brief CUDA error checking macro
 *
 * Evaluates a CUDA runtime call and checks for errors.
 * If an error is detected, prints detailed information and terminates the program.
 */
#define CUDA_CHECK(call) \
do { \
    cudaError_t error = call; \
    if (error != cudaSuccess) { \
        std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__ << " - " \
        << cudaGetErrorString(error) << " (" << error << ") " << std::endl; \
        exit(EXIT_FAILURE); \
} \
}while(0)

/**
 * @brief Initialise multiple arrays with random values in a specified range
 *
 * @param arrays     Array of pointers to initialize
 * @param num_arrays Number of arrays to initialize
 * @param size       Number of elements in each array
 * @param min       Minimum value for random numbers (default: 0.0)
 * @param max       Maximum value for random numbers (default: 1.0)
 * @param seed       Seed for random generator, 0 means use time(0) (default: 0)
 */
void initialiseArrays(float** arrays, int num_arrays, size_t size, float min=0.0f, float max=1.0f, unsigned int seed=0)
{
    // Set random seed
    if (seed == 0)
    {
        seed = static_cast<unsigned int>(time(0)); // get current time
    }
    srand(seed);

    float range = max - min;

    for (int i = 0; i < num_arrays; i++) // Iterate through each array pointer
    {
        for (size_t j = 0; j < size; j++) // Iterate through each element
        {
            arrays[i][j] = min + (static_cast<float>(rand()) / RAND_MAX) * range;
        }
    }
}

/**
 * @brief Measure CPU execution time using std::chrono
 *
 * @tparam Func     Function type
 * @param function  Function or lambda to measure
 * @return double   Execution time in milliseconds
 */
template <typename Function>
double measureExecutionTime(Function function)
{
    auto start = std::chrono::steady_clock::now();
    function();
    auto end = std::chrono::steady_clock::now();
    std::chrono::duration<double, std::milli> duration = (end - start);
    return duration.count();
}

/**
 * @brief Measure GPU kernel execution time using CUDA events
 *
 * @tparam KernelFunc  Kernel function type
 * @tparam Args        Kernel argument types
 * @param kernel       Kernel function to measure
 * @param grid         Grid dimensions
 * @param block        Block dimensions
 * @param args         Kernel arguments
 * @return float       Execution time in milliseconds
 */
template <typename KernelFunc>
float measureKernelTime(KernelFunc kernel)
{
    cudaEvent_t start;
    cudaEvent_t stop;
    float elapsed_time;

    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    // Start stopwatch
    CUDA_CHECK(cudaEventRecord(start));
    // Launch kernel
    kernel();
    // Stop stopwatch
    CUDA_CHECK(cudaEventRecord(stop));

    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&elapsed_time, start, stop));

    // Free events
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    return elapsed_time;
}

/**
 * @brief Compare results between CPU and GPU outputs using absolute and relative tolerance
 *
 * @param cpu_result  CPU computed results
 * @param gpu_result  GPU computed results
 * @param size        Number of elements to compare
 * @param atol        Absolute tolerance (default: 1e-4)
 * @param rtol        Relative tolerance (default: 1e-5)
 * @return bool       True if results match within tolerances, false otherwise
 */
bool compareResults(const float *cpu_result, const float *gpu_result,
                    size_t size, float atol = 1e-4f, float rtol = 1e-5f)
{
    for (size_t i = 0; i < size; i++)
    {
        float a = cpu_result[i];
        float b = gpu_result[i];
        float abs_diff = std::fabs(a - b);
        float rel_diff = abs_diff / std::fmax(std::fabs(a), std::fabs(b));

        if (abs_diff > atol && rel_diff > rtol)
        {
            std::cout << "Mismatch at index " << i
                      << ": CPU = " << a
                      << ", GPU = " << b
                      << ", abs diff = " << abs_diff
                      << ", rel diff = " << rel_diff
                      << std::endl;
            return false;
        }
    }
    return true;
}

#endif


Overwriting utils.h


In [34]:
%%writefile conv_1d.cu
#include <cuda_bf16.h>
#include <cuda_fp16.h>
#include <cuda_fp8.h>
#include <cuda_runtime.h>
#include <float.h>
#include <stdio.h>
#include <stdlib.h>
#include <cmath>
#include <cstddef>
#include <iostream>
#include "utils.h"


__global__ void naive_conv_1d(const float* x, const float* f, float* y, int f_w, int N) {
    int t_idx = blockIdx.x * blockDim.x + threadIdx.x;
    int radius = (f_w - 1) / 2;
    float res = 0.0f;

    if (t_idx >= N) return;

    for (int i = 0; i < f_w; i++) {
        int global_idx = t_idx - radius + i;
        float val = (global_idx >= 0 && global_idx < N) ? x[global_idx] : 0.0f;
        res += val * f[i];
    }

    y[t_idx] = res;
}

void conv1d_cpu_ref(const float* x, const float* f, float* y, int N, int f_w) {
    const int radius = (f_w - 1) / 2;

    for (int t = 0; t < N; ++t) {
        float acc = 0.0f;

        for (int i = 0; i < f_w; ++i) {
            const int xi = t - radius + i;              // input index
            const float xv = (xi >= 0 && xi < N) ? x[xi] : 0.0f;
            acc += xv * f[i];
        }

        y[t] = acc;
    }
}

int main() {
    int N = 1 << 20;
    int f_w = 5;

    // Calculate size of 1D vector
    size_t size_x = N * sizeof(float);
    // Calculate size of 1D convolution filter
    size_t size_f = f_w * sizeof(float);
    size_t size_y = N * sizeof(float);

    // Create host pointers and allocate host memory
    float* X_host = (float*)malloc(size_x);
    float* F_host = (float*)malloc(size_f);
    float* Y_host_cpu = (float*)malloc(size_y);
    float* Y_host_gpu = (float*)malloc(size_y);

    // Initialise the arrays 
    float* array_X[] {X_host};
    float* array_F[] {X_host};
    initialiseArrays(array_X, 1, N, 0.0f, 100.0f, 0);
    initialiseArrays(array_F, 1, f_w, 1.0f, 50.0f, 0);

    // Measure CPU reference time
    float cpu_time = measureExecutionTime([&](){
        conv1d_cpu_ref(X_host, F_host, Y_host_cpu, N, f_w);
    });

    std::cout << "CPU execution time: " << cpu_time << "ms" << std::endl;

    // Create device ptrs and allocate device memory
    float* X_device;
    float* F_device; 
    float* Y_device;

    cudaMalloc((void**)&X_device, size_x);
    cudaMalloc((void**)&F_device, size_f);
    cudaMalloc((void**)&Y_device, size_y);

    // Copy arrays to device memory
    cudaMemcpy(X_device, X_host, size_x, cudaMemcpyHostToDevice);
    cudaMemcpy(F_device, F_host, size_f, cudaMemcpyHostToDevice);

    // Configure grid and block dims
    int num_blocks = (N + 32 - 1 ) / 32;
    dim3 blockDim(32);
    dim3 gridDim(num_blocks);

    // Execute kernel & measure time
    double gpu_time = measureKernelTime([&]() {
        naive_conv_1d<<<gridDim, blockDim>>>(X_device, F_device, Y_device, f_w, N);
        cudaDeviceSynchronize();
    });
    
    std::cout << "GPU execution time: " << gpu_time << "ms" << std::endl;
    std::cout << "Speedup: " << cpu_time / gpu_time << "x" << std::endl;

    // Copy results back to host
    cudaMemcpy(Y_host_gpu, Y_device, size_y, cudaMemcpyDeviceToHost);

    // Compare against cpu reference for correctness
    bool results_match = compareResults(Y_host_cpu, Y_host_gpu, N, 1e-4, 1e-5);
    std::cout << (results_match ? "Results match!" : "Results do not match!") << std::endl;

    // Free memories
    free(X_host);
    free(F_host);
    free(Y_host_cpu);
    free(Y_host_gpu);

    cudaFree(X_device);
    cudaFree(F_device);
    cudaFree(Y_device);
    
    return 0;
}

Overwriting conv_1d.cu


In [36]:
!nvcc -O2 -arch=sm_70 conv_1d.cu -o conv_1d
!./conv_1d

CPU execution time: 10.623ms
GPU execution time: 0.22272ms
Speedup: 47.6967x
Results match!
